# Exercises 2 — Experimental Design & Offline Protocols (Student)

In [2]:
# ============================================================
# Part 0 — Environment & Dataset
# ============================================================
!pip -q install gradio gensim sentence-transformers transformers

from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import math, random, warnings
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize, StandardScaler
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import roc_auc_score, mean_absolute_error, mean_squared_error

warnings.filterwarnings("ignore")

BASE = Path("/content/drive/MyDrive/Colab Notebooks")
DATA_ROOT = BASE / "Dataset_Chapter3_4"

if (DATA_ROOT / "data").exists():
    DATA_DIR = DATA_ROOT / "data"
else:
    DATA_DIR = DATA_ROOT

print("BASE      =", BASE)
print("DATA_ROOT =", DATA_ROOT)
print("DATA_DIR  =", DATA_DIR)

items = pd.read_csv(DATA_DIR / "metadata.csv")
ratings = pd.read_csv(DATA_DIR / "ratings.csv")

rename_items = {
    "movie_id":"item_id", "id":"item_id",
    "movie_title":"title", "name":"title",
    "genres":"genre", "description":"synopsis",
    "plot":"synopsis", "summary":"synopsis"
}
for old,new in rename_items.items():
    if old in items.columns and new not in items.columns:
        items = items.rename(columns={old:new})

rename_r = {"uid":"user_id","user":"user_id","iid":"item_id","movie_id":"item_id","score":"rating"}
for old,new in rename_r.items():
    if old in ratings.columns and new not in ratings.columns:
        ratings = ratings.rename(columns={old:new})

required_i = ["item_id","title","genre","synopsis"]
required_r = ["user_id","item_id","rating"]
assert all(c in items.columns for c in required_i), items.columns.tolist()
assert all(c in ratings.columns for c in required_r), ratings.columns.tolist()

items["item_id"] = items["item_id"].astype(str)
ratings["item_id"] = ratings["item_id"].astype(str)
ratings["user_id"] = ratings["user_id"].astype(str)
ratings["rating"] = pd.to_numeric(ratings["rating"])

items["genre"] = items["genre"].fillna("").astype(str)
items["synopsis"] = items["synopsis"].fillna("").astype(str)
items["text_content"] = (items["genre"] + " " + items["synopsis"]).str.strip()

ITEM_INDEX = {iid:i for i,iid in enumerate(items["item_id"])}
N_USERS = max(1, ratings["user_id"].nunique())
POPULARITY = ratings.groupby("item_id")["user_id"].nunique().to_dict()

print(f"Items={len(items)}, Users={ratings['user_id'].nunique()}, Ratings={len(ratings)}")
display(items[["item_id","title","genre","synopsis"]].head())


# ============================================================
# Core Functions
# Vector Transformation + 3 User Profiles + Ranking + Metrics
# ============================================================

LIKED_THRESHOLD = 4.0
DISLIKED_THRESHOLD = 2.0

def transform_vectors(X, mode="Raw+L2"):
    X = np.asarray(X, dtype=float)
    if mode == "Raw+L2":
        return normalize(X)
    if mode == "Center+L2":
        return normalize(X - X.mean(axis=0, keepdims=True))
    if mode == "Standard+L2":
        return normalize(StandardScaler().fit_transform(X))
    raise ValueError(mode)

def _unit(v):
    v=np.asarray(v,dtype=float)
    n=np.linalg.norm(v)
    return v if n==0 else v/n

def profile_positive_only(user_id, X, ratings_df):
    r=ratings_df[ratings_df["user_id"].astype(str)==str(user_id)]
    liked=r[r["rating"]>=LIKED_THRESHOLD]
    idx=[ITEM_INDEX[str(i)] for i in liked["item_id"] if str(i) in ITEM_INDEX]
    if not idx:
        idx=[ITEM_INDEX[str(i)] for i in r["item_id"] if str(i) in ITEM_INDEX]
    return _unit(X[idx].mean(axis=0)) if idx else np.zeros(X.shape[1])

def profile_rating_centered(user_id, X, ratings_df):
    r=ratings_df[ratings_df["user_id"].astype(str)==str(user_id)].copy()
    r=r[r["item_id"].astype(str).isin(ITEM_INDEX)]
    if r.empty: return np.zeros(X.shape[1])
    idx=np.array([ITEM_INDEX[str(i)] for i in r["item_id"]])
    w=r["rating"].astype(float).values - float(r["rating"].mean())
    denom=np.abs(w).sum()
    p=X[idx].mean(axis=0) if denom<=1e-12 else (X[idx]*w[:,None]).sum(axis=0)/denom
    return _unit(p)

def profile_rocchio(user_id, X, ratings_df, alpha=1.0, beta=1.0):
    r=ratings_df[ratings_df["user_id"].astype(str)==str(user_id)]
    liked=r[r["rating"]>=LIKED_THRESHOLD]
    disliked=r[r["rating"]<=DISLIKED_THRESHOLD]
    li=[ITEM_INDEX[str(i)] for i in liked["item_id"] if str(i) in ITEM_INDEX]
    di=[ITEM_INDEX[str(i)] for i in disliked["item_id"] if str(i) in ITEM_INDEX]
    pos=X[li].mean(axis=0) if li else np.zeros(X.shape[1])
    neg=X[di].mean(axis=0) if di else np.zeros(X.shape[1])
    p=alpha*pos-beta*neg
    if np.linalg.norm(p)==0:
        hi=[ITEM_INDEX[str(i)] for i in r["item_id"] if str(i) in ITEM_INDEX]
        p=X[hi].mean(axis=0) if hi else p
    return _unit(p)

PROFILE_METHODS = {
    "Positive-only": profile_positive_only,
    "Rating-centered": profile_rating_centered,
    "Rocchio": profile_rocchio,
}

def get_profile(user_id, X, ratings_df, method, alpha=1.0, beta=1.0):
    if method=="Rocchio":
        return profile_rocchio(user_id,X,ratings_df,alpha,beta)
    return PROFILE_METHODS[method](user_id,X,ratings_df)

def rank_user(user_id, X, ratings_df, method, k=None, candidates=None, alpha=1.0, beta=1.0):
    p=get_profile(user_id,X,ratings_df,method,alpha,beta)
    score=cosine_similarity(p.reshape(1,-1),X).ravel()
    seen=set(ratings_df.loc[ratings_df["user_id"].astype(str)==str(user_id),"item_id"].astype(str))
    if candidates is None:
        candidates=[iid for iid in items["item_id"] if iid not in seen]
    ranked=sorted(candidates,key=lambda iid:score[ITEM_INDEX[iid]],reverse=True)
    return ranked[:k] if k else ranked, score, p

# Accuracy / Ranking Metrics
def precision_at_k(recs, rel, k):
    return len(set(recs[:k]) & set(rel))/k if k else 0.0
def recall_at_k(recs, rel, k):
    rel=set(rel); return len(set(recs[:k])&rel)/len(rel) if rel else 0.0
def f1_at_k(recs, rel, k):
    p=precision_at_k(recs,rel,k); r=recall_at_k(recs,rel,k)
    return 0.0 if p+r==0 else 2*p*r/(p+r)
def hr_at_k(recs, rel, k):
    return float(bool(set(recs[:k])&set(rel)))
def reciprocal_rank(recs, rel):
    rel=set(rel)
    for rank,iid in enumerate(recs,1):
        if iid in rel: return 1.0/rank
    return 0.0
def ap_at_k(recs, rel, k):
    rel=set(rel); hits=0; s=0.0
    for i,iid in enumerate(recs[:k],1):
        if iid in rel:
            hits+=1; s+=hits/i
    return s/min(len(rel),k) if rel else 0.0
def ndcg_at_k(recs, rel, k):
    rel=set(rel)
    dcg=sum(1/math.log2(i+2) for i,iid in enumerate(recs[:k]) if iid in rel)
    idcg=sum(1/math.log2(i+2) for i in range(min(len(rel),k)))
    return dcg/idcg if idcg else 0.0

# Beyond-Accuracy Metrics
def coverage(all_recs):
    return len(set(all_recs))/len(items) if len(items) else 0.0

def gini_coefficient(all_recs):
    counts=pd.Series(all_recs).value_counts()
    x=np.array([counts.get(iid,0) for iid in items["item_id"]],dtype=float)
    if x.sum()==0: return 0.0
    x=np.sort(x); n=len(x)
    return float((2*np.sum(np.arange(1,n+1)*x)/(n*x.sum()))-(n+1)/n)

def ild(item_ids, X):
    idx=[ITEM_INDEX[i] for i in item_ids if i in ITEM_INDEX]
    if len(idx)<2: return 0.0
    D=1-cosine_similarity(X[idx])
    v=D[np.triu_indices_from(D,k=1)]
    return float(v.mean()) if len(v) else 0.0

def novelty(item_ids):
    vals=[]
    for iid in item_ids:
        pop=max(1,POPULARITY.get(iid,1))
        vals.append(-math.log2(pop/N_USERS))
    return float(np.mean(vals)) if vals else 0.0

def calibration_genre(user_id, rec_ids, ratings_df):
    hist=ratings_df[ratings_df["user_id"].astype(str)==str(user_id)]
    hist_genres=[]
    for iid in hist["item_id"].astype(str):
        if iid in ITEM_INDEX:
            hist_genres += [g.strip() for g in str(items.iloc[ITEM_INDEX[iid]]["genre"]).replace("|",",").split(",") if g.strip()]
    rec_genres=[]
    for iid in rec_ids:
        rec_genres += [g.strip() for g in str(items.iloc[ITEM_INDEX[iid]]["genre"]).replace("|",",").split(",") if g.strip()]
    if not hist_genres or not rec_genres: return np.nan
    cats=sorted(set(hist_genres)|set(rec_genres))
    hp=np.array([hist_genres.count(c) for c in cats],float); hp/=hp.sum()
    rp=np.array([rec_genres.count(c) for c in cats],float); rp/=rp.sum()
    return float(1-0.5*np.abs(hp-rp).sum())

def loo_folds(ratings_df, positive_threshold=4.0):
    folds=[]
    for uid,g in ratings_df.groupby("user_id"):
        pos=g[g["rating"]>=positive_threshold]
        if pos.empty: continue
        hold=pos.sort_values("rating").iloc[-1]
        folds.append((str(uid),ratings_df.drop(index=hold.name),str(hold["item_id"])))
    return folds

def evaluate_config(X, profile_method, k=3, alpha=1.0, beta=1.0, folds=None,
                    candidate_mode="Full Catalog", n_neg=20, seed=42):
    folds=folds or loo_folds(ratings)
    rng=np.random.default_rng(seed)
    rows=[]; global_recs=[]
    for uid,train,heldout in folds:
        seen=set(train.loc[train["user_id"].astype(str)==uid,"item_id"].astype(str))
        all_c=[iid for iid in items["item_id"] if iid not in seen]
        if heldout not in all_c: all_c.append(heldout)
        if candidate_mode=="Sampled Negatives":
            neg=[x for x in all_c if x!=heldout]
            sampled=list(rng.choice(neg,size=min(n_neg,len(neg)),replace=False)) if neg else []
            candidates=[heldout]+sampled
        else:
            candidates=all_c
        ranked,score,_=rank_user(uid,X,train,profile_method,candidates=candidates,alpha=alpha,beta=beta)
        topk=ranked[:k]; global_recs += topk
        y=np.array([1 if iid==heldout else 0 for iid in ranked])
        ys=np.array([score[ITEM_INDEX[iid]] for iid in ranked])
        auc=roc_auc_score(y,ys) if len(np.unique(y))==2 else np.nan
        rows.append({
            "user_id":uid,"heldout":heldout,
            f"Precision@{k}":precision_at_k(topk,[heldout],k),
            f"Recall@{k}":recall_at_k(topk,[heldout],k),
            f"F1@{k}":f1_at_k(topk,[heldout],k),
            f"HR@{k}":hr_at_k(topk,[heldout],k),
            "MRR":reciprocal_rank(ranked,[heldout]),
            f"MAP@{k}":ap_at_k(topk,[heldout],k),
            f"nDCG@{k}":ndcg_at_k(topk,[heldout],k),
            "AUC":auc,"ILD":ild(topk,X),"Novelty":novelty(topk),
            "Calibration":calibration_genre(uid,topk,train),
            "TopK":" | ".join(topk)
        })
    detail=pd.DataFrame(rows)
    summary=detail.select_dtypes(include=np.number).mean().to_dict() if len(detail) else {}
    summary["Coverage"]=coverage(global_recs)
    summary["Gini"]=gini_coefficient(global_recs)
    return pd.Series(summary), detail

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.8/27.8 MB 45.4 MB/s eta 0:00:00
Mounted at /content/drive
BASE      = /content/drive/MyDrive/Colab Notebooks
DATA_ROOT = /content/drive/MyDrive/Colab Notebooks/Dataset_Chapter3_4
DATA_DIR  = /content/drive/MyDrive/Colab Notebooks/Dataset_Chapter3_4/data
Items=12, Users=8, Ratings=32


,item_id,title,genre,synopsis
0,i1,Toy Story 5,Animation|Family,A cowboy toy and a space ranger toy lead the t...
1,i2,Inside Out 2,Animation|Family,"Inside the mind of a teenage girl, five emotio..."
2,i3,The Wild Robot,Animation|Sci-Fi,A shipwrecked robot boots up on a wild island ...
3,i4,It Ends with Us,Drama|Romance,A florist falls in love with a charming surgeo...
4,i5,Wicked,Fantasy|Musical,A green-skinned woman with a gift for magic jo...


In [15]:
tfidf=TfidfVectorizer(lowercase=True,ngram_range=(1,2))

X=transform_vectors(tfidf.fit_transform(items["text_content"]).toarray(),"Raw+L2")
folds=loo_folds(ratings)

## ข้อ 1 — Candidate Protocol
ใช้ representation, profile แบบ Rating-centered และ folds เดียวกัน เปรียบเทียบ `Full Catalog` กับ `Sampled Negatives`  กรณี `Sampled Negatives` กำหนด negative sample 5

In [4]:
# TODO: เรียก evaluate_config() สำหรับ candidate_mode 2 ค่า
rows=[]
for cand in ["Full Catalog","Sampled Negatives"]:
    s,_=evaluate_config(X,"Rating-centered",k=3,folds=folds,candidate_mode=cand,n_neg=5)
    rows.append({"Candidates":cand,**s.to_dict()})
display(pd.DataFrame(rows).round(4))

,Candidates,Precision@3,Recall@3,F1@3,HR@3,MRR,MAP@3,nDCG@3,AUC,ILD,Novelty,Calibration,Coverage,Gini
0,Full Catalog,0.2083,0.625,0.3125,0.625,0.6958,0.6250,0.6250,0.7969,0.9360,1.5236,0.4479,0.9167,0.2986
1,Sampled Negatives,0.2500,0.750,0.3750,0.750,0.7292,0.6667,0.6875,0.8000,0.9397,1.7390,0.4688,1.0000,0.2431


` 1.1  เรียก evaluate_config()
*   Sampled Negatives ---  n_neg = 5, k=5

*   Compared Item Vector Creation with"TF-IDF","Word2Vec","SBERT", Normalized with "Raw+L2"
*  User Profile with
    "Rating-centered", "Rocchio"
    




In [20]:
# TF-IDF
tfidf=TfidfVectorizer(lowercase=True,ngram_range=(1,2),min_df=1)
X_tfidf=tfidf.fit_transform(items["text_content"]).toarray()

# Word2Vec
from gensim.models import Word2Vec
tokens=[str(x).lower().split() for x in items["text_content"]]
w2v=Word2Vec(tokens,vector_size=100,window=5,min_count=1,workers=2,epochs=100,seed=42)
def doc_w2v(tok):
    vv=[w2v.wv[w] for w in tok if w in w2v.wv]
    return np.mean(vv,axis=0) if vv else np.zeros(w2v.vector_size)
X_w2v=np.vstack([doc_w2v(t) for t in tokens])

# Sentence-BERT
from sentence_transformers import SentenceTransformer
sbert=SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
X_sbert=sbert.encode(items["text_content"].tolist(),show_progress_bar=True)

# BERT mean pooling
import torch
from transformers import AutoTokenizer, AutoModel
bert_name="bert-base-uncased"
bert_tok=AutoTokenizer.from_pretrained(bert_name)
bert=AutoModel.from_pretrained(bert_name); bert.eval()
def bert_embed(texts,batch_size=8):
    rows=[]
    for i in range(0,len(texts),batch_size):
        b=texts[i:i+batch_size]
        inp=bert_tok(b,padding=True,truncation=True,max_length=128,return_tensors="pt")
        with torch.no_grad():
            h=bert(**inp).last_hidden_state
        mask=inp["attention_mask"].unsqueeze(-1)
        z=(h*mask).sum(1)/mask.sum(1).clamp(min=1)
        rows.append(z.numpy())
    return np.vstack(rows)
X_bert=bert_embed(items["text_content"].tolist())

RAW_REPRESENTATIONS={
    "TF-IDF":X_tfidf,
    "Word2Vec":X_w2v,
    "SBERT":X_sbert
}
for name,X in RAW_REPRESENTATIONS.items(): print(name,X.shape)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


TF-IDF (12, 800)
Word2Vec (12, 100)
SBERT (12, 384)


In [21]:
REP_TO_RUN=["TF-IDF","Word2Vec","SBERT"]

PROFILES_TO_RUN=list(PROFILE_METHODS)

CANDIDATES_TO_RUN=["Full Catalog","Sampled Negatives"]

rows=[]
for rep_name in REP_TO_RUN:

        X=transform_vectors(RAW_REPRESENTATIONS[rep_name],"Raw+L2")
        for prof in PROFILES_TO_RUN:

                    s,_=evaluate_config(
                        X,prof,k=3,
                        candidate_mode="Sampled Negatives",n_neg=20,seed=42)
                    rows.append({
                        "Representation":rep_name,"Transformation":trans,
                        "Profile":prof,"Candidates":"Sampled Negatives",
                        **s.to_dict()
                    })
experiment_results=pd.DataFrame(rows)
display(experiment_results.round(4))

,Representation,Transformation,Profile,Candidates,Precision@3,Recall@3,F1@3,HR@3,MRR,MAP@3,nDCG@3,AUC,ILD,Novelty,Calibration,Coverage,Gini
0,TF-IDF,Center+L2,Positive-only,Sampled Negatives,0.2917,0.875,0.4375,0.875,0.6597,0.6458,0.7039,0.7946,0.9347,1.5896,0.5104,0.8333,0.3819
1,TF-IDF,Center+L2,Rating-centered,Sampled Negatives,0.2083,0.625,0.3125,0.625,0.6958,0.6250,0.6250,0.7969,0.9360,1.5236,0.4479,0.9167,0.2986
2,TF-IDF,Center+L2,Rocchio,Sampled Negatives,0.2083,0.625,0.3125,0.625,0.6292,0.5625,0.5789,0.7634,0.9387,1.5479,0.4271,0.9167,0.2639
3,Word2Vec,Center+L2,Positive-only,Sampled Negatives,0.2083,0.625,0.3125,0.625,0.6095,0.5417,0.5625,0.7411,0.0001,1.6108,0.4115,0.7500,0.5278
4,Word2Vec,Center+L2,Rating-centered,Sampled Negatives,0.2083,0.625,0.3125,0.625,0.5139,0.4375,0.4866,0.7361,0.0001,1.5896,0.4063,1.0000,0.1875
5,Word2Vec,Center+L2,Rocchio,Sampled Negatives,0.2083,0.625,0.3125,0.625,0.5139,0.4375,0.4866,0.7321,0.0001,1.5306,0.3854,1.0000,0.2014
6,SBERT,Center+L2,Positive-only,Sampled Negatives,0.2500,0.750,0.3750,0.750,0.7264,0.6875,0.7039,0.7969,0.6763,1.6069,0.4896,1.0000,0.2917
7,SBERT,Center+L2,Rating-centered,Sampled Negatives,0.2917,0.875,0.4375,0.875,0.5625,0.5417,0.6280,0.8237,0.6895,1.7217,0.3646,1.0000,0.2708
8,SBERT,Center+L2,Rocchio,Sampled Negatives,0.2500,0.750,0.3750,0.750,0.5312,0.4792,0.5491,0.7902,0.6641,1.6211,0.3490,1.0000,0.2847


## ข้อ 2 — K Sensitivity
ทดลอง K=1,3,5 แล้วเปรียบเทียบ Precision, Recall, HR, nDCG และ Coverage

In [13]:
# TODO
rows=[]
for k in [1,3,5]:
    s,_=evaluate_config(X,"Rating-centered",k=k,folds=folds)
    rows.append({"K":k,**s.to_dict()})
display(pd.DataFrame(rows).round(4))

,K,Precision@1,Recall@1,F1@1,HR@1,MRR,MAP@1,nDCG@1,AUC,ILD,...,F1@3,HR@3,MAP@3,nDCG@3,Precision@5,Recall@5,F1@5,HR@5,MAP@5,nDCG@5
0,1,0.625,0.625,0.625,0.625,0.6958,0.625,0.625,0.7969,0.0000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,3,NaN,NaN,NaN,NaN,0.6958,NaN,NaN,0.7969,0.9360,...,0.3125,0.625,0.625,0.625,NaN,NaN,NaN,NaN,NaN,NaN
2,5,NaN,NaN,NaN,NaN,0.6958,NaN,NaN,0.7969,0.9433,...,NaN,NaN,NaN,NaN,0.175,0.875,0.2917,0.875,0.675,0.7217


## ข้อ 3 — Profile × Transformation
สร้าง experiment table 2 transformations × 3 profiles

In [17]:
# TODO
rows=[]
raw =tfidf.fit_transform(items["text_content"]).toarray()
for trans in ["Raw+L2","Center+L2"]:
    Xt=transform_vectors(raw,trans)
    for prof in PROFILE_METHODS:
        s,_=evaluate_config(Xt,prof)
        rows.append({"Transformation":trans,"Profile":prof,**s.to_dict()})
display(pd.DataFrame(rows).round(4))

,Transformation,Profile,Precision@3,Recall@3,F1@3,HR@3,MRR,MAP@3,nDCG@3,AUC,ILD,Novelty,Calibration,Coverage,Gini
0,Raw+L2,Positive-only,0.2917,0.875,0.4375,0.875,0.6597,0.6458,0.7039,0.7946,0.9347,1.5896,0.5104,0.8333,0.3819
1,Raw+L2,Rating-centered,0.2083,0.625,0.3125,0.625,0.6958,0.6250,0.6250,0.7969,0.9360,1.5236,0.4479,0.9167,0.2986
2,Raw+L2,Rocchio,0.2083,0.625,0.3125,0.625,0.6292,0.5625,0.5789,0.7634,0.9387,1.5479,0.4271,0.9167,0.2639
3,Center+L2,Positive-only,0.2917,0.875,0.4375,0.875,0.7604,0.7292,0.7664,0.9062,1.1003,1.6729,0.6146,0.8333,0.3819
4,Center+L2,Rating-centered,0.2083,0.625,0.3125,0.625,0.6958,0.6250,0.6250,0.7969,1.0895,1.5236,0.4479,0.9167,0.2986
5,Center+L2,Rocchio,0.2083,0.625,0.3125,0.625,0.6292,0.5625,0.5789,0.7634,1.0950,1.5896,0.4479,0.9167,0.2431


## ข้อ 4 — Design Question
ถ้าต้องการพิสูจน์ว่า SBERT ดีกว่า TF-IDF ควรควบคุมอะไรให้เหมือนกันบ้าง?

In [23]:
rows=[]
for rep_name in ["TF-IDF","SBERT"]:
    X = transform_vectors(RAW_REPRESENTATIONS[rep_name],"Center+L2")
    s,_=evaluate_config(X,"Rating-centered")
    rows.append({"transform_method":rep_name,**s.to_dict()})
display(pd.DataFrame(rows).round(4))

,transform_method,Precision@3,Recall@3,F1@3,HR@3,MRR,MAP@3,nDCG@3,AUC,ILD,Novelty,Calibration,Coverage,Gini
0,TF-IDF,0.2083,0.625,0.3125,0.625,0.6958,0.6250,0.625,0.7969,1.0895,1.5236,0.4479,0.9167,0.2986
1,SBERT,0.2917,0.875,0.4375,0.875,0.5625,0.5417,0.628,0.8237,1.0277,1.7217,0.3646,1.0000,0.2708
